# Classification Reliability Heat Maps

Two label-retention reliability checks for the primary pathway
classification (see `gbctp.reliability` for the implementation; this
notebook only configures inputs and plots the results):

- **Part A -- Bootstrap convergence.** Thresholds are learned from a random
  subset of `n` pathways (`n` swept from 10% to 100% of the dataset) and
  applied to the full dataset. Tests sensitivity to *which* pathways were
  used to define the thresholds.
- **Part B -- Combined bootstrap + threshold perturbation.** As above, but
  the quantile levels themselves (e.g. the 75th percentile) are additionally
  perturbed by a few percentile points, and the recovery cutoffs are
  jittered too. A stronger test of sensitivity to the exact threshold
  *definitions*.

For a class c, the retention score at sample size n is the fraction of
reference-class-c pathways that keep the same primary label across bootstrap
(and, for Part B, threshold-perturbation) repeats. Values close to 1 mean
that class's membership is essentially unaffected by finite sampling (and,
for Part B, by the exact threshold definitions).

## Part A -- Bootstrap convergence heat map

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gbctp.reliability import bootstrap_classification_convergence

# ---------------------------------------------------------------------------
# INPUT DATA
#
# ORDER_DESCRIPTOR_FILE is the paths_order_descriptors_with_archetype_labels.csv
# written by 01_pathway_classification.ipynb (gbctp.pipeline.run_pipeline).
# ---------------------------------------------------------------------------
ROOT_DIR = Path("<path/to/output_dir>")  # TODO: same OUTPUT_DIR as notebook 01
ORDER_DESCRIPTOR_FILE = ROOT_DIR / "paths_order_descriptors_with_archetype_labels.csv"

OUT_DIR = Path("<path/to/results_dir>")  # TODO: where to write figures/CSVs
OUT_DIR.mkdir(exist_ok=True, parents=True)

INCLUDE_DIFFUSE_AS_5TH_LABEL = False
N_BOOTSTRAP = 300
SAMPLE_FRACTIONS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
RANDOM_SEED = 42

PRIMARY_ORDER = [
    "Source-limited",
    "GB-crossing (recovering)",
    "GB-blocked (non-recovering)",
    "Target-limited",
]
if INCLUDE_DIFFUSE_AS_5TH_LABEL:
    PRIMARY_ORDER = ["Diffuse/weakly structured"] + PRIMARY_ORDER

required_cols = [
    "pE", "E_dip", "C_dip", "rJ", "rC", "Lweak_E", "Lweak_C",
    "drop_E", "rise_E", "drop_C", "rise_C", "RE",
]
df = pd.read_csv(ORDER_DESCRIPTOR_FILE)
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required descriptor columns: {missing}")

stability_df = bootstrap_classification_convergence(
    df,
    sample_fractions=SAMPLE_FRACTIONS,
    n_bootstrap=N_BOOTSTRAP,
    random_seed=RANDOM_SEED,
    primary_order=PRIMARY_ORDER,
    include_diffuse_as_5th_label=INCLUDE_DIFFUSE_AS_5TH_LABEL,
)
stability_df.to_csv(OUT_DIR / "classification_convergence_values.csv", index=False)
print(stability_df.round(3).to_string(index=False))


In [ ]:
def plot_convergence_heatmap(stability_df, primary_order, out_dir):
    plot_df = stability_df.set_index("n_sampled_paths")[primary_order]

    fig_width = max(8, 1.8 * len(primary_order))
    fig_height = max(5, 0.45 * len(plot_df))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    im = ax.imshow(plot_df.values, aspect="auto", vmin=0.0, vmax=1.0)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Primary-label retention fraction")

    ax.set_title("Bootstrap convergence of pathway classification", pad=14)
    ax.set_xlabel("Reference primary label")
    ax.set_ylabel("Number of pathways used to define thresholds")
    ax.set_xticks(np.arange(len(plot_df.columns)))
    ax.set_xticklabels(plot_df.columns, rotation=35, ha="right")
    ax.set_yticks(np.arange(len(plot_df.index)))
    ax.set_yticklabels(plot_df.index)

    for i in range(plot_df.shape[0]):
        for j in range(plot_df.shape[1]):
            value = plot_df.iloc[i, j]
            if not np.isnan(value):
                ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=8)

    fig.tight_layout()
    fig.savefig(out_dir / "classification_convergence_heatmap.png", dpi=300, bbox_inches="tight")
    fig.savefig(out_dir / "classification_convergence_heatmap.pdf", bbox_inches="tight")
    plt.show()


plot_convergence_heatmap(stability_df, PRIMARY_ORDER, OUT_DIR)


## Part B -- Combined bootstrap + threshold-perturbation heat map

In [ ]:
from gbctp.reliability import run_combined_bootstrap_threshold_analysis

# Threshold perturbations per bootstrap sample, and their widths.
N_PERTURB = 30
Q_PERTURB_WIDTH = 5.0    # e.g. q75 sampled from [q70, q80]
REC_PERTURB_WIDTH = 0.02  # rec_hi = 1.05 +/- this, rec_lo = 0.95 +/- this

# Sample sizes to sweep (absolute pathway counts, not fractions, for Part B).
MAX_SAMPLE_SIZE = 500
SAMPLE_STEP = 50

stability_mean_df, stability_std_df, all_results_df = run_combined_bootstrap_threshold_analysis(
    df,
    n_bootstrap=N_BOOTSTRAP,
    n_perturb=N_PERTURB,
    random_seed=RANDOM_SEED,
    primary_order=PRIMARY_ORDER,
    q_perturb_width=Q_PERTURB_WIDTH,
    rec_perturb_width=REC_PERTURB_WIDTH,
    max_sample_size=MAX_SAMPLE_SIZE,
    sample_step=SAMPLE_STEP,
    include_diffuse_as_5th_label=INCLUDE_DIFFUSE_AS_5TH_LABEL,
)

stability_mean_df.to_csv(OUT_DIR / "combined_stability_mean_matrix.csv")
stability_std_df.to_csv(OUT_DIR / "combined_stability_std_matrix.csv")
all_results_df.to_csv(OUT_DIR / "bootstrap_threshold_all_retention_values.csv", index=False)

print("Mean stability over all classes:")
print(stability_mean_df.mean(axis=1).round(3).to_string())


### Paper figure: combined stability heat map

In [ ]:
# Maps full column names (e.g. "Source-limited") to short x-axis tick labels.
COLUMN_LABEL_MAP = {
    "Source-limited": "SL",
    "GB-crossing (recovering)": "GBC",
    "GB-blocked (non-recovering)": "GBB",
    "Target-limited": "TL",
    "Diffuse/weakly structured": "DIF",
}


def plot_combined_stability_heatmap(
    stability_mean_df, stability_std_df=None, cmap="viridis", fig_size=8,
    out_name="combined_bootstrap_threshold_stability_heatmap",
    show_std=False, column_label_map=COLUMN_LABEL_MAP,
):
    plot_df = stability_mean_df.copy()
    nrows, ncols = plot_df.shape

    fig, ax = plt.subplots(figsize=(fig_size, fig_size))

    data_vals = plot_df.values.astype(float)
    valid_vals = data_vals[~np.isnan(data_vals)]
    data_min, data_max = np.min(valid_vals), np.max(valid_vals)
    pad = 0.01 * (data_max - data_min) if data_max > data_min else 0.01
    vmin, vmax = max(0.0, data_min - pad), min(1.0, data_max + pad)

    im = ax.imshow(plot_df.values, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    cbar = fig.colorbar(im, ax=ax, fraction=0.05, pad=0.04)
    cbar.set_label("Primary-label retention fraction")

    ax.set_xlabel("Primary labels", labelpad=10)
    ax.set_ylabel("Sampled pathways (N)", labelpad=10)

    xtick_labels = [column_label_map.get(c, c) for c in plot_df.columns] if column_label_map else list(plot_df.columns)
    ax.set_xticks(np.arange(ncols))
    ax.set_xticklabels(xtick_labels)
    ax.tick_params(axis="x", bottom=False, top=False)
    ax.set_yticks(np.arange(nrows))
    ax.set_yticklabels(plot_df.index)

    midpoint = (vmin + vmax) / 2.0
    for i in range(nrows):
        for j in range(ncols):
            mean_val = plot_df.iloc[i, j]
            if np.isnan(mean_val):
                text, color = "NA", "black"
            else:
                if show_std and stability_std_df is not None and not np.isnan(stability_std_df.iloc[i, j]):
                    text = f"{mean_val:.2f}\n\u00b1{stability_std_df.iloc[i, j]:.2f}"
                else:
                    text = f"{mean_val:.2f}"
                color = "white" if mean_val < midpoint else "black"
            ax.text(j, i, text, ha="center", va="center", fontsize=9, color=color)

    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{out_name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{out_name}.pdf", bbox_inches="tight")
    plt.show()


plot_combined_stability_heatmap(stability_mean_df, stability_std_df, cmap="viridis", show_std=False)
